# 04 — Capacity Analysis

Regional generation capacity vs demand, interconnector utilisation, fuel mix trends.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from uk_energy.config import *

## 1. Load Plants

In [ ]:
if PLANTS_UNIFIED.exists():
    df = pd.read_parquet(PLANTS_UNIFIED)
    operational = df[df['status'] == 'operational'].copy()
    print(f'Total plants: {len(df):,}')
    print(f'Operational: {len(operational):,}')
    total_gw = operational['capacity_mw'].sum() / 1000
    print(f'Total operational capacity: {total_gw:.1f} GW')
else:
    print('No plants data — run the pipeline first')

## 2. Capacity by Fuel Type

In [ ]:
if 'operational' in dir():
    fuel_cap = (
        operational.groupby('fuel_type')['capacity_mw']
        .sum()
        .sort_values(ascending=True)
    )
    
    fig = px.bar(
        x=fuel_cap.values / 1000,
        y=fuel_cap.index,
        orientation='h',
        title='Installed Capacity by Fuel Type (GW)',
        labels={'x': 'Capacity (GW)', 'y': 'Fuel Type'},
    )
    fig.show()

## 3. Regional Generation vs Estimated Demand

In [ ]:
# Load DNO region reference for demand estimates
with open(DNO_REGIONS_REF) as f:
    dno_data = json.load(f)

# Approximate peak demand by region (GW) - rough estimates
region_demand_approx = {
    'North Scotland': 1.5,
    'South Scotland': 2.5,
    'North West England': 5.0,
    'North East England': 3.0,
    'Yorkshire': 4.5,
    'North Wales & Mersey': 2.5,
    'South Wales': 2.0,
    'West Midlands': 5.5,
    'East Midlands': 4.5,
    'East England': 4.0,
    'South West England': 3.5,
    'South England': 3.5,
    'London': 8.5,
    'South East England': 4.5,
}
print('Approximate regional demand estimates loaded')

In [ ]:
if 'operational' in dir():
    regional_gen = (
        operational.groupby('dno_region')['capacity_mw']
        .sum()
        .reset_index()
        .rename(columns={'capacity_mw': 'generation_mw'})
    )
    regional_gen['demand_mw'] = regional_gen['dno_region'].map(
        lambda r: region_demand_approx.get(r, 2000) * 1000
    )
    regional_gen['gen_vs_demand'] = regional_gen['generation_mw'] / regional_gen['demand_mw']
    
    fig = go.Figure(data=[
        go.Bar(name='Generation Capacity (MW)', x=regional_gen['dno_region'], y=regional_gen['generation_mw']),
        go.Bar(name='Peak Demand Estimate (MW)', x=regional_gen['dno_region'], y=regional_gen['demand_mw']),
    ])
    fig.update_layout(
        title='Regional Generation Capacity vs Demand',
        barmode='group',
        xaxis_tickangle=-45,
        height=500,
    )
    fig.show()
    display(regional_gen.sort_values('gen_vs_demand', ascending=False))

## 4. Interconnector Dependency

In [ ]:
with open(INTERCONNECTORS_REF) as f:
    ic_data = json.load(f)

ic_df = pd.DataFrame(ic_data['interconnectors'])
total_ic_mw = ic_df['capacity_mw'].sum()

fig = px.sunburst(
    ic_df,
    path=['countries', 'id'],
    values='capacity_mw',
    title=f'UK Interconnector Capacity by Country ({total_ic_mw:,} MW total)',
)
fig.show()

print(f'\nTotal interconnector capacity: {total_ic_mw:,} MW ({total_ic_mw/1000:.1f} GW)')

## 5. Current Generation Mix (Carbon Intensity API)

In [ ]:
from uk_energy.ingest.carbon_intensity import parse_generation_mix

mix_df = parse_generation_mix()
if not mix_df.empty:
    fig = px.pie(
        mix_df,
        names='fuel',
        values='perc',
        title='Current UK Generation Mix',
        hole=0.4,
    )
    fig.show()
    display(mix_df.sort_values('perc', ascending=False))